# 🧠 DyslexiaLens — Data Science Notebook
## Persiapan Dataset & Preprocessing

**Tema Capstone:** Accessible & Adaptive Learning  
**Peran:** Data Scientist  
**Dataset:** Gambo — Dyslexia Handwriting Dataset (Kaggle)

---

### Deskripsi Singkat
Notebook ini mencakup seluruh proses persiapan data (*Data Wrangling*) untuk proyek **DyslexiaLens**, sebuah sistem *early screening* disleksia berbasis analisis citra tulisan tangan.

Sistem ini **bukan alat diagnosis medis**, melainkan alat bantu skrining awal yang memberikan indikasi dini kepada orang tua dan pendidik sebelum pemeriksaan lebih lanjut oleh profesional.

---

### Alur Notebook
| Tahap | Deskripsi | Sifat |
|---|---|---|
| **Tahap 1** | Assessing Data — Evaluasi kualitas & struktur dataset, audit distribusi dan anomali | Wajib |
| **Tahap 2** | Physical Renaming — Normalisasi skala nama file fisik ke skala 1–6 | Opsional |
| **Tahap 3** | Cleaning Data & CSV Generation — Filter data kotor, buat master CSV | Wajib |
| **Tahap 4** | *(coming soon)* EDA & Visualisasi Data | Wajib |

## 📌 Pertanyaan Bisnis

Notebook ini dirancang untuk menjawab dua pertanyaan bisnis utama yang terukur:

**1. Dapatkah pola goresan tulisan tangan digunakan sebagai indikator awal disleksia?**
> Hipotesis: Tulisan dari penderita disleksia memiliki pola goresan yang secara visual dapat dibedakan dari tulisan normal — baik melalui pembalikan arah huruf (*Reversal*) maupun koreksi berulang yang meninggalkan bekas coretan (*Corrected*).

**2. Seberapa parah tingkat gejala disleksia berdasarkan keparahan coretan (Severity Score 0–6)?**
> Hipotesis: Tingkat keparahan goresan dapat dikuantifikasi secara bertingkat, di mana skor lebih tinggi merepresentasikan distorsi tulisan yang lebih ekstrem dan lebih kuat mengindikasikan gejala disleksia.

---

### Definisi Variabel Target

| Variabel | Tipe | Nilai | Keterangan |
|---|---|---|---|
| `target_class` | Integer | `0` = Normal, `1` = Disleksia | Label biner untuk klasifikasi utama |
| `severity_score` | Integer | `0` = Sehat, `1`–`6` = Ringan hingga Parah | Skor keparahan gejala disleksia |
| `folder_category` | String | `Normal`, `Corrected`, `Reversal` | Kategori kelas asal dari dataset |

> **⚠️ Catatan Anti Data Leakage:** Variabel `severity_score` dan `folder_category` **tidak boleh dimasukkan sebagai fitur input model** karena merupakan turunan langsung dari label. Hanya `image_path` yang menjadi input — dan `target_class` yang menjadi output target utama.

## 📦 Sumber Dataset & Referensi Ilmiah

### Identitas Dataset
| Atribut | Detail |
|---|---|
| **Nama** | Dyslexia Handwriting Dataset (Gambo) |
| **Sumber** | Kaggle — Dataset Publik |
| **Link** | https://www.kaggle.com/datasets/drizasazanitaisa/dyslexia-handwriting-dataset |
| **Total Gambar** | 208.372 file `.png` |
| **Format** | Grayscale, resolusi 28×28 piksel |
| **Kelas** | `Normal`, `Corrected`, `Reversal` |

### Cara Mendapatkan Dataset
Dataset tidak disertakan di dalam repositori Git (di-*gitignore*) karena ukurannya yang besar.  
Untuk menjalankan notebook ini secara **lokal**, lakukan langkah berikut:
1. Unduh dataset dari link Kaggle di atas.
2. Ekstrak dan letakkan folder `Gambo/` di direktori yang sama dengan file notebook ini.
3. Pastikan struktur folder: `Gambo/Train/` dan `Gambo/Test/` sudah tersedia.

Untuk menjalankan di **Google Colab**:
1. Upload folder `Gambo/` ke Google Drive Anda.
2. Mount Google Drive di Colab, lalu ubah variabel `root_dir` di setiap cell menjadi path Drive Anda (misal: `'/content/drive/MyDrive/Gambo'`).

### Referensi Ilmiah
Dataset ini dipublikasikan dan digunakan dalam penelitian berikut:

1. M. S. A. B. Rosli, I. S. Isa, S. A. Ramlan, S. N. Sulaiman and M. I. F. Maruzuki, *"Development of CNN Transfer Learning for Dyslexia Handwriting Recognition"*, 2021 11th IEEE International Conference on Control System, Computing and Engineering (ICCSCE), 2021, pp. 194-199, doi: 10.1109/ICCSCE52189.2021.9530971.

2. N. S. L. Seman, I. S. Isa, S. A. Ramlan, W. Li-Chih and M. I. F. Maruzuki, *"Classification of Handwriting Impairment Using CNN for Potential Dyslexia Symptom"*, 2021 11th IEEE International Conference on Control System, Computing and Engineering (ICCSCE), 2021, pp. 188-193, doi: 10.1109/ICCSCE52189.2021.9530989.

3. Isa, Iza Sazanita. *CNN Comparisons Models On Dyslexia Handwriting Classification*. Universiti Teknologi MARA Cawangan Pulau Pinang, 2021.

4. Isa, I. S., Rahimi, W. N. S., Ramlan, S. A., & Sulaiman, S. N. (2019). *Automated detection of dyslexia symptom based on handwriting image for primary school children*. Procedia Computer Science, 163, 440-449.

---
# Tahap 1: Assessing Data — Evaluasi Kualitas & Struktur Dataset

Sebelum melakukan pembersihan data, kita perlu memahami **apa yang ada di dalam dataset** secara menyeluruh. Tahap *Assessing Data* ini meliputi:
1. Mengecek distribusi jumlah file per kelas dan per split
2. Memvalidasi konsistensi format (ekstensi, resolusi, mode warna)
3. Mengaudit penamaan file untuk menemukan anomali label

---

### Temuan Utama (Hasil Audit Manual + Algoritmik)

**✅ Temuan 1 — Sistem Severity Score (Kunci)**  
Folder numerik `1, 4, 5, 6, 7, 8, 9` di dalam kelas `Corrected` dan `Reversal` **bukan** merepresentasikan karakter angka yang ditulis, melainkan **Tingkat Keparahan Goresan (Severity Score)**. Inspeksi visual membuktikan hampir seluruh abjad tersebar merata di tiap folder — tidak ada konsentrasi karakter tertentu. Ini berarti dataset **tidak mengalami Data Leakage** dan model akan dipaksa belajar pola goresan, bukan menghafal karakter.

| Skor Asli (Periset) | Kondisi Visual | Keparahan Nyata | Skor AI (Setelah Normalisasi) |
|---|---|---|---|
| `1` | Goresan hancur lebur, hampir tidak terbaca | **Paling Parah** | `6` |
| `4` | Banyak timpa-menimpa, kerangka huruf samar | Parah | `6` (digabung) |
| `5`–`8` | Koreksi tampak, huruf masih bisa ditebak | Menengah | `5` → `2` |
| `9` | Goresan ringan, huruf aslinya mudah dikenali | **Paling Ringan** | `1` |

> ⚠️ Skala asli bersifat **terbalik** — `1` = Parah (bukan Ringan), `9` = Ringan (bukan Parah). Jika langsung digunakan sebagai label angka, model AI akan menganggap tulisan paling parah (skor 1) justru paling dekat dengan Normal (skor 0). Ini dikoreksi melalui *Dictionary Mapping* di Tahap 2 & Tahap 3.

---

**⚠️ Temuan 2 — Kontaminasi Label (Label Noise) di Kelas Non-Normal**  
Ditemukan file bernama `NormalXXXX.png` yang terselip di dalam folder `Corrected` dan `Reversal`. Secara visual, konten gambar-gambar tersebut adalah goresan cacat — bukan tulisan tangan normal yang bersih. Ini adalah *Label Noise* yang terjadi kemungkinan akibat *script rename otomatis* tanpa verifikasi visual dari pihak pembuat dataset asli.

---

**⚠️ Temuan 3 — Anomali Visual di Kelas Normal Asli**  
Melalui inspeksi visual manual pada sampel gambar di dalam kelas `Normal`, ditemukan bahwa **tidak semua gambar di kelas Normal benar-benar bersih**. Sejumlah sampel menampilkan goresan koreksi atau pola yang secara visual menyerupai karakteristik `Corrected`, meskipun secara nama file dan letak folder dikategorikan sebagai `Normal`.

Ini memperkuat indikasi adanya **Kontaminasi Label Dua Arah** dari dataset asli:
- File bergoresan cacat masuk ke folder Normal (`NormalXXXX.png` tersesat)
- File bertulisan normal yang secara visual wajar, namun labelnya terkontaminasi oleh goresan lingkungan sekitar saat pengambilan data

> 📌 Temuan ini menjadi dasar keputusan untuk **tidak semata-mata mengandalkan nama folder** sebagai label tunggal, dan mendukung penggunaan `master_dataset_dyslexia.csv` dengan filter eksplisit sebagai sumber kebenaran (*ground truth*) untuk proses training.

### 1A. Statistik Distribusi Dataset

In [ ]:
import os
from pathlib import Path
from collections import defaultdict

root_dir = r'Gambo'

stats = defaultdict(lambda: defaultdict(int))
total = 0
noise_count = 0
non_png_count = 0

for root, dirs, files in os.walk(root_dir):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext != '.png':
            non_png_count += 1
            continue
        parts = Path(root).parts
        try:
            split_type = parts[-2]
            category  = parts[-1]
        except:
            continue
        stats[split_type][category] += 1
        total += 1
        if 'Normal' in file and category != 'Normal':
            noise_count += 1

print('=' * 55)
print('     STATISTIK DISTRIBUSI DATASET GAMBO')
print('=' * 55)
for split, categories in sorted(stats.items()):
    subtotal = sum(categories.values())
    print(f'\n📁 {split} ({subtotal:,} gambar)')
    for cat, count in sorted(categories.items()):
        print(f'   └── {cat:12s}: {count:>7,} gambar')

print(f'\n{"-" * 55}')
print(f'  Total file .png          : {total:,}')
print(f'  File non-.png ditemukan  : {non_png_count:,}  (diabaikan)')
print(f'  File Noise terdeteksi    : {noise_count:,}  (akan di-drop di Tahap 3)')
print(f'  Estimasi data bersih     : {total - noise_count:,}')
print('=' * 55)

### 1B. Verifikasi Integritas File, Resolusi & Format Warna

Kode berikut melakukan *sampling* acak dari setiap kelas untuk memverifikasi:
- Apakah file dapat dibuka (tidak corrupt)
- Konsistensi resolusi (ukuran piksel)
- Konsistensi format warna (*color mode*)

In [ ]:
import os
import random
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

root_dir = r'Gambo'
SAMPLE_PER_CLASS = 500

class_files = defaultdict(list)
for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.png'):
            parts = Path(root).parts
            try:
                category = parts[-1]
            except:
                continue
            class_files[category].append(os.path.join(root, file))

print('=' * 55)
print('  VERIFIKASI INTEGRITAS, RESOLUSI & FORMAT WARNA')
print('=' * 55)

for category, paths in sorted(class_files.items()):
    sample = random.sample(paths, min(SAMPLE_PER_CLASS, len(paths)))
    resolutions = Counter()
    modes = Counter()
    corrupt = 0
    for path in sample:
        try:
            with Image.open(path) as img:
                resolutions[img.size] += 1
                modes[img.mode] += 1
        except Exception:
            corrupt += 1

    print(f'\n📂 Kelas: {category} (sample {len(sample)} dari {len(paths):,} file)')
    print(f'   File corrupt        : {corrupt}')
    print(f'   Resolusi ditemukan  : {dict(resolutions.most_common(3))}')
    print(f'   Format warna        : {dict(modes)}')

print('\n' + '=' * 55)
print('  Verifikasi selesai.')
print('=' * 55)

### Kesimpulan Assessing

| Aspek | Status | Keterangan |
|---|---|---|
| Struktur folder | ✅ | `Train/Test → Normal/Corrected/Reversal` — hierarki bersih |
| Format file | ✅ | 100% berformat `.png`, nol file non-PNG |
| Integritas file | ✅ | Nol file corrupt dari hasil sampling |
| Resolusi gambar | ✅ | 28×28 piksel (identik dengan format MNIST/EMNIST) |
| Format warna | ✅ | Grayscale (`L`) atau Binary (`1`) — tidak ada RGB |
| Distribusi kelas | ⚠️ | Corrected lebih dominan — perlu pantau *class imbalance* |
| Severity Score | ✅ | Folder 1–9 = derajat keparahan (bukan karakter), skala terbalik |
| Label Noise (kelas non-Normal) | ⚠️ | File `NormalXXXX.png` terselip — di-*drop* di Tahap 3 |
| Anomali visual kelas Normal | ⚠️ | Sebagian sampel Normal mengandung goresan tidak wajar — tercatat sebagai risiko residual |

---
# Tahap 2: Preprocessing Tambahan (Physical Renaming) — Opsional
Kode opsional ini secara fisik mengganti nama file di local disk Anda agar skala keparahannya berurutan menjadi skala **1 sampai 6** (1 = Paling Ringan, 6 = Paling Parah) menggunakan *Dictionary Mapping*:

| Skor Asli | → | Skor Baru | Keterangan |
|---|---|---|---|
| `9` | → | `1` | Paling Ringan |
| `8` | → | `2` | |
| `7` | → | `3` | |
| `6` | → | `4` | |
| `5` | → | `5` | |
| `4` | → | `6` | Corrected Paling Parah |
| `1` | → | `6` | Reversal (digabung ke puncak) |

> ⚠️ **Jalankan sel ini hanya SEKALI.** Menjalankannya dua kali pada dataset yang sudah di-rename akan menyebabkan skala bergeser dan data menjadi kacau.
>
> ℹ️ **Opsional:** Jika Anda melewati Tahap 2, kode Tahap 3 tetap berfungsi karena `get_score` sudah dirancang untuk menangani **kedua kondisi** (skala asli dan skala setelah di-rename).

In [11]:
import os
import time

root_dir = r'Gambo'

print("Mempersiapkan penggantian nama (Physical Renaming) skala 1-6...")

score_map = {
    9: 1, 8: 2, 7: 3, 6: 4, 5: 5,
    4: 6, # Asli 4 (Corrected Paling Parah) -> Jadi Skor AI 6
    1: 6  # Asli 1 (Reversal) -> Sama-sama disatukan di puncak Skor AI 6
}

to_rename = []
for root, dirs, files in os.walk(root_dir):
    if not ('Corrected' in root or 'Reversal' in root):
        continue
    for file in files:
        if file.endswith('.png'):
            prefix, separator = None, None
            if '_' in file:
                head = file.split('_')[0]
                if head.isdigit() and int(head) in score_map:
                    prefix, separator = int(head), '_'
            elif '-' in file:
                head = file.split('-')[0]
                if head.isdigit() and int(head) in score_map:
                    prefix, separator = int(head), '-'
            if prefix is not None:
                new_prefix = score_map[prefix]
                sisa_nama = file.split(separator, 1)[1]
                new_name = f"{new_prefix}{separator}{sisa_nama}"
                to_rename.append((os.path.join(root, file), os.path.join(root, new_name)))

print(f"Total file yang akan dimapping ulang namanya: {len(to_rename)} file.")

temp_rename = []
for old, new in to_rename:
    temp_path = new + ".TEMP"
    os.rename(old, temp_path)
    temp_rename.append((temp_path, new))

for temp, final_new in temp_rename:
    berhasil = False
    for _ in range(10):
        try:
            os.replace(temp, final_new)
            berhasil = True
            break
        except PermissionError:
            time.sleep(0.1)
    if not berhasil:
        print(f"Gagal: {final_new}")

print("Selesai! Seluruh nama fisik file berhasil dipetakan ke skala konsisten (1 Ringan -> 6 Parah)!")

Mempersiapkan penggantian nama (Physical Renaming) skala 1-6...
Total file yang akan dimapping ulang namanya: 121835 file.
Selesai! Seluruh nama fisik file berhasil dipetakan ke skala konsisten (1 Ringan -> 6 Parah)!


---
# Tahap 3: Cleaning Data & Pembuatan Master CSV

Kode ini melakukan **Logical Cleaning** — memfilter semua file kotor (*Label Noise*) langsung dari tabel Pandas **tanpa menghapus file fisik aslinya**, lalu menyimpan daftar data bersih ke `master_dataset_dyslexia.csv`.

Fungsi `get_score` dirancang untuk bekerja dalam **dua kondisi secara otomatis**:
- **Jika Tahap 2 dijalankan:** nama file sudah berprefix 1–6, kode membaca langsung.
- **Jika Tahap 2 dilewati:** nama file masih berprefix asli (1, 4, 5, 6, 7, 8, 9), kode menerapkan *Dictionary Mapping* secara otomatis ke rentang 0–6.

> **INGAT:** Ubah variabel `root_dir` sesuai lokasi dataset Anda (lokal atau Google Drive/Colab).

In [12]:
import os
import pandas as pd
from pathlib import Path

root_dir = r'Gambo'

# Dictionary Mapping: Skor asli periset (terbalik) -> Skor AI (linear 0-6)
# Digunakan sebagai fallback jika Tahap 2 (Physical Renaming) dilewati.
SCORE_MAP_ORIGINAL = {1: 6, 4: 6, 5: 5, 6: 4, 7: 3, 8: 2, 9: 1}

print("Membaca seluruh direktori...")
data = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith('.png'):
            parts = Path(root).parts
            try:
                split_type = parts[-2]
                category = parts[-1]
            except:
                continue
            path_full = os.path.join(root, file)
            data.append({
                'image_path': path_full,
                'file_name': file,
                'split': split_type,
                'folder_category': category
            })

df = pd.DataFrame(data)
print(f"Total gambar berserakan yang ditemukan: {len(df)} file.\n")

def get_score(row):
    """Ekstrak & normalisasi Severity Score dari nama file.
    Bekerja otomatis untuk dataset pre-Tahap 2 (skor asli) maupun post-Tahap 2 (skor 1-6).
    """
    filename = row['file_name']
    folder = row['folder_category']

    # Buang file NormalXXXX yang tersesat ke kelas non-Normal (Label Noise)
    if 'Normal' in filename and folder != 'Normal':
        return 'DROP'

    # Kelas Normal -> Skor 0 (Sehat)
    if folder == 'Normal':
        return 0

    # Kelas Corrected & Reversal -> Ekstrak prefix angka
    if folder in ['Corrected', 'Reversal']:
        for sep in ['_', '-']:
            if sep in filename:
                prefix = filename.split(sep)[0]
                if prefix.isdigit():
                    val = int(prefix)
                    if 1 <= val <= 6:
                        # Post-Tahap 2: nama file sudah di-rename ke skala 1-6
                        return val
                    elif val in SCORE_MAP_ORIGINAL:
                        # Pre-Tahap 2: terapkan Dictionary Mapping ke skala AI 0-6
                        return SCORE_MAP_ORIGINAL[val]
                break

    return 'DROP'  # Buang file tak dikenal

df['severity_score'] = df.apply(get_score, axis=1)
df_clean = df[~df['severity_score'].astype(str).str.contains('DROP')].copy()
df_clean['severity_score'] = df_clean['severity_score'].astype(int)
df_clean['target_class'] = df_clean['severity_score'].apply(lambda x: 0 if x == 0 else 1)

output_csv = 'master_dataset_dyslexia.csv'
df_clean.to_csv(output_csv, index=False)

print(f"SUCCESS! Disimpan ke '{output_csv}' sebanyak {len(df_clean)} file gambar bersih.")
df_clean.sample(5)

Membaca seluruh direktori...
Total gambar berserakan yang ditemukan: 208372 file.

SUCCESS! Disimpan ke 'master_dataset_dyslexia.csv' sebanyak 180726 file gambar bersih.
